Apache Beam notebook outlines the requirements for processing loan data using Apache Beam, focusing on identifying loan defaulters based on specific criteria for Personal and Medical loan categories.

### Loan File Key Points:

*   **Personal Loan Category:**
    *   Bank does not accept short or late payments.
    *   If a person has not paid a monthly installment, that month's entry will not be present in the file.

*   **Medical Loan Category:**
    *   Bank accepts late payments, but the payment must be the full amount.
    *   It is assumed that there is a data/record for every month for Medical Loans.

### Loan Defaulters Criteria:

*   **Medical Loan Defaulters:**
    *   If a customer has made a total of 3 or more late payments.

*   **Personal Loan Defaulters:**
    *   If a customer has missed a total of 4 or more installments **OR**
    *   If a customer has missed 2 consecutive installments.

In [ ]:
!pip install --quiet apache-beam[gcp]

In [ ]:
from google.colab import files
files.upload()

In [13]:
from apache_beam.io.textio import WriteToText
import apache_beam as beam
p1 = beam.Pipeline()

def MedicalLoanDefaulters(element):
  # Input: A single string representing a row from the loan file.
  # Example input: 'CT88330,Humberto,Banks,Serviceman,LN_1559,Medical Loan,26-01-2018, 2000, 30-01-2018'
  customer_id = element.split(',')[0]
  loan_category = element.split(',')[5]
  payment_date = element.split(',')[8]
  due_date = element.split(',')[6]
  # Output: A tuple (customer_id : loan_category, default_indicator)
  # default_indicator is 1 if payment_date > due_date, else 0.
  # Example output: ('CT88330 : Medical Loan', 1) or ('CT88330 : Medical Loan', 0)
  if payment_date > due_date:
    return customer_id+" : "+loan_category, 1
  else:
    return customer_id+" : "+loan_category, 0


class PersonalLoanDefaulter(beam.DoFn):
  # The process method receives an element from the PCollection.
  # Input: A tuple (customer_id, iterable_of_payment_dates) from beam.GroupByKey.
  # Example input: ('CT88330', [' 30-01-2018', ' 28-02-2018', ' 28-03-2018'])
  def process(self,element):
    customer_id,payment_dates = element
    month_list=list()
    prev_value=0
    for dat in payment_dates:
      month = int(dat.split('-')[1])
      if month-prev_value>=2: # Check for 2 consecutive missed installments
        # Output: A list containing a single tuple (customer_id : Personal Loan, missed_count)
        # Example output: [('CT88330 : Personal Loan', 2)]
        return [(customer_id+" : Personal Loan",month-prev_value)]

      month_list.append(month)
      prev_value=month

    if len(month_list)>0:
      # Check for 4 or more missed installments
      # Note: This logic assumes month_list is sorted, which it is not automatically.
      # If `month_list` was populated from sorted `payment_dates`, it might be sorted.
      missed_payment=set(range(month_list[0],month_list[-1])) - set(month_list)
      if len(missed_payment)>4:
        # Output: A list containing a single tuple (customer_id : Personal Loan, missed_count)
        # Example output: [('CT88330 : Personal Loan', 5)]
        return [(customer_id+" : Personal Loan",len(missed_payment))]



read_pcoll = (p1
    |"Read loan file">>beam.io.ReadFromText('/content/loan.txt',skip_header_lines=1))
# Input: Each line of the '/content/loan.txt' file as a string (header skipped).
# Example: 'CT88330,Humberto,Banks,Serviceman,LN_1559,Medical Loan,26-01-2018, 2000, 30-01-2018'


medical_loan_defaulters = (
    read_pcoll
    # Filters the PCollection to include only 'Medical Loan' entries.
    # Input: String (loan record). Output: String (medical loan record).
    |"filter medical loans">>beam.Filter(lambda element:element.split(',')[5]=="Medical Loan")
    # Transforms each medical loan record into a (customer_id : loan_category, default_indicator) tuple.
    # Input: String (medical loan record). Output: Tuple (key, value).
    |"map rows">>beam.Map(MedicalLoanDefaulters)
    # Groups elements by key (customer_id : loan_category) and sums their default_indicators.
    # Input: PCollection of (key, value) tuples. Output: PCollection of (key, sum_of_values).
    # Example input: ('CT88330 : Medical Loan', 1), ('CT88330 : Medical Loan', 0), ('CT88330 : Medical Loan', 1)
    # Example output: ('CT88330 : Medical Loan', 2)
    |"combine per customer_id">>beam.CombinePerKey(sum)
    # Filters to keep only customers with 3 or more late payments (medical loan defaulters).
    # Input: Tuple (customer_id : loan_category, total_late_payments). Output: Same tuple if total_late_payments >= 3.
    |"filter defaulter customer">>beam.Filter(lambda element:element[1]>=3)
    # |"Write to File">>beam.io.WriteToText('/content/loan-defaulters.txt')
)

personal_loan_defaulters = (
    read_pcoll
    # Filters the PCollection to include only 'Personal Loan' entries.
    # Input: String (loan record). Output: String (personal loan record).
    |"Filter personal loans">>beam.Filter(lambda element:element.split(',')[5]=="Personal Loan")
    # Extracts customer ID and payment date, forming a (customer_id, payment_date_string) tuple.
    # Input: String (personal loan record).
    # Output: Tuple (customer_id, payment_date_string).
    # Example input: 'CT88330,Humberto,Banks,Serviceman,LN_1559,Personal Loan,26-01-2018, 2000, 30-01-2018'
    # Example output: ('CT88330', ' 30-01-2018')
    |"Map to customer_id,payment_date for Personal Loan">>beam.Map(lambda element: ((element.split(',')[0]), (element.split(',')[8])))
    # Groups payment dates by customer ID.
    # Input: PCollection of (customer_id, payment_date_string) tuples.
    # Output: PCollection of (customer_id, iterable_of_payment_date_strings) tuples.
    # Example input: ('CT88330', ' 30-01-2018'), ('CT88330', ' 28-02-2018')
    # Example output: ('CT88330', [' 30-01-2018', ' 28-02-2018', ' 28-03-2018', ...])
    |"Group by customer-id and payment_date">>beam.GroupByKey()
    # Applies the PersonalLoanDefaulter DoFn to identify personal loan defaulters.
    # Input: PCollection of (customer_id, iterable_of_payment_date_strings) tuples.
    # Output: PCollection of (customer_id : Personal Loan, missed_installments_count) tuples.
    # Example output: ('CT88330 : Personal Loan', 4) or ('CT88330 : Personal Loan', 2)
    |"Personal loan defaulter logic">>beam.ParDo(PersonalLoanDefaulter())

)

#Combine Medical & Personal Loan Defaulter list
combined_pcoll = ((medical_loan_defaulters,personal_loan_defaulters)
                  |"combine loan defaulters">>beam.Flatten()
                  |"Write to output file">>beam.io.WriteToText('/content/loan-defaulters.txt')

)

p1.run()

!{('head -n 10 /content/loan-def*')}

('CT71222 : Medical Loan', 9)
('CT14299 : Medical Loan', 6)
('CT63122 : Medical Loan', 5)
('CT12439 : Medical Loan', 7)
('CT95698 : Medical Loan', 7)
('CT82062 : Medical Loan', 5)
('CT83820 : Medical Loan', 10)
('CT93771 : Medical Loan', 7)
('CT79071 : Medical Loan', 7)
('CT75832 : Medical Loan', 8)


In [ ]:
!pip install langchain openai

In [6]:
customer_id,payment_dates=('CT56276', ['23-01-2019', '17-02-2019'])
month_list=list()
for dat in payment_dates:
      month_list.append(int(dat.split('-')[1]))
        # month_list.append(dat.split('-')[1])

print(month_list)
range(month_list[0],month_list[-1])
missed_payment=set(range(month_list[0],month_list[-1])) - set(month_list)
print(len(missed_payment))

[1, 2]
0
